In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score

# Load the datasets
df = pd.read_csv("DataSet.csv")
test_df = pd.read_csv("Test_DataSet.csv")

# Encode the Season as a numerical value
season_encoder = LabelEncoder()
all_seasons = pd.concat([df['season'], test_df['season']]).dropna().unique()
season_encoder.fit(all_seasons)
df["Season_Encoded"] = season_encoder.transform(df["season"])
test_df["Season_Encoded"] = season_encoder.transform(test_df["season"])

# Extract year and month
df["Year"] = df["month"].apply(lambda x: int(x.split("_")[1]))
df["Month"] = df["month"].apply(lambda x: int(x.split("_")[0]))
test_df["Year"] = test_df["month"].apply(lambda x: int(x.split("_")[1]))
test_df["Month"] = test_df["month"].apply(lambda x: int(x.split("_")[0]))

# Pivot precipitation data
pivot_df = df.pivot_table(index=["Longitude", "Latitude"], columns="Month", values="Precipitation_Cleaned").reset_index()
pivot_df.columns = ["Longitude", "Latitude"] + [f"Month_{i}" for i in sorted(df["Month"].unique())]

test_pivot_df = test_df.pivot_table(index=["Longitude", "Latitude"], columns="Month", values="Precipitation_Cleaned").reset_index()
test_pivot_df.columns = ["Longitude", "Latitude"] + [f"Month_{i}" for i in sorted(test_df["Month"].unique())]

# Merge season information
season_data = df.groupby(["Longitude", "Latitude"])['Season_Encoded'].agg(lambda x: x.mode()[0]).reset_index()
test_season_data = test_df.groupby(["Longitude", "Latitude"])['Season_Encoded'].agg(lambda x: x.mode()[0]).reset_index()

final_df = pivot_df.merge(season_data, on=["Longitude", "Latitude"])
final_test_df = test_pivot_df.merge(test_season_data, on=["Longitude", "Latitude"])

# Calculate total precipitation
train_month_cols = [col for col in pivot_df.columns if col.startswith("Month_")]
test_month_cols = [col for col in test_pivot_df.columns if col.startswith("Month_")]

final_df["Total_Precipitation"] = final_df[train_month_cols].sum(axis=1)
final_test_df["Total_Precipitation"] = final_test_df[test_month_cols].sum(axis=1)

# Define rainfall categories
low_threshold = final_df["Total_Precipitation"].quantile(0.33)
high_threshold = final_df["Total_Precipitation"].quantile(0.66)

def categorize_rainfall(value):
    if value <= low_threshold:
        return "Low Rainfall"
    elif value <= high_threshold:
        return "Medium Rainfall"
    else:
        return "High Rainfall"

final_df["Rainfall_Category"] = final_df["Total_Precipitation"].apply(categorize_rainfall)
final_test_df["Rainfall_Category"] = final_test_df["Total_Precipitation"].apply(categorize_rainfall)

# Encode categories
category_mapping = {"Low Rainfall": 0, "Medium Rainfall": 1, "High Rainfall": 2}
final_df["Rainfall_Category"] = final_df["Rainfall_Category"].map(category_mapping)
final_test_df["Rainfall_Category"] = final_test_df["Rainfall_Category"].map(category_mapping)

# Select common features
common_features = ["Longitude", "Latitude", "Season_Encoded"] + list(set(train_month_cols) & set(test_month_cols))

# Features and labels
X_train = final_df[common_features]
y_train = final_df["Rainfall_Category"]
X_test = final_test_df[common_features]
y_test = final_test_df["Rainfall_Category"]

# Train Decision Tree Model
dt_model = DecisionTreeClassifier(max_depth=10, min_samples_split=5, random_state=42)
dt_model.fit(X_train, y_train)

# Training Predictions
y_train_pred = dt_model.predict(X_train)
y_test_pred = dt_model.predict(X_test)

# Model Evaluation
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
cross_val_scores = cross_val_score(dt_model, X_train, y_train, cv=5)

print(f"\nTraining Accuracy: {train_accuracy:.2f}")
print(f"Test Accuracy: {test_accuracy:.2f}")
print(f"Cross-Validation Accuracy: {np.mean(cross_val_scores):.2f} ± {np.std(cross_val_scores):.2f}")

print("\nClassification Report (Test Data):\n", classification_report(y_test, y_test_pred))

# Confusion Matrix
plt.figure(figsize=(6, 4))
conf_matrix = confusion_matrix(y_test, y_test_pred)
sns.heatmap(conf_matrix, annot=True, cmap="Blues", fmt="d",
            xticklabels=["Low", "Medium", "High"],
            yticklabels=["Low", "Medium", "High"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# Distribution Visualization
plt.figure(figsize=(8, 5))
sns.histplot(final_df["Total_Precipitation"], bins=30, kde=True, label="Training Data", alpha=0.5)
sns.histplot(final_test_df["Total_Precipitation"], bins=30, kde=True, label="Test Data", alpha=0.5)
plt.axvline(low_threshold, color='r', linestyle='dashed', label='Low Threshold')
plt.axvline(high_threshold, color='g', linestyle='dashed', label='High Threshold')
plt.xlabel("Total Precipitation")
plt.ylabel("Frequency")
plt.title("Precipitation Data Distribution (Training vs Test)")
plt.legend()
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/Test_DataSet.csv'